# Gold Layer: Dimensional Modeling

Builds Gold datasets from cleaned Silver tables in `dev.silver`:
- **dim_geography** — one row per nation, including parent region
- **dim_customer** — one row per customer with geography
- **dim_product** — one row per product
- **dim_supplier** — one row per supplier with geography
- **fact_sales** — one row per order line, with measures and denormalized attributes
- **vw_annual_sales_kpis** — annual revenue, orders, active customers, and average order value

Includes reconciliation of Gold revenue and row count against Silver, plus dimension key uniqueness validation.

In [0]:
# ── Setup & Config ──────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")
SILVER_SCHEMA = "silver"
GOLD_SCHEMA  = "gold"

print(f"✅ Catalog: {CATALOG} | Silver: {CATALOG}.{SILVER_SCHEMA} | Gold: {CATALOG}.{GOLD_SCHEMA}")

In [0]:
# ── dim_geography: one row per nation, including parent region ──
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.dim_geography AS
SELECT
    n.n_nationkey       AS geography_key,
    n.n_nationkey       AS nation_key,
    n.n_name            AS nation_name,
    n.n_regionkey       AS region_key,
    r.r_name            AS region_name,
    n.n_comment         AS nation_comment,
    r.r_comment         AS region_comment
FROM {CATALOG}.{SILVER_SCHEMA}.nation n
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.region r
    ON n.n_regionkey = r.r_regionkey
""")

_row_count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_geography").count()
print(f"dim_geography created: {_row_count} rows")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_geography"))

In [0]:
# ── dim_customer: one row per customer, with geography ──
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.dim_customer AS
SELECT
    c.c_custkey         AS customer_key,
    c.c_name            AS customer_name,
    c.c_address         AS customer_address,
    c.c_nationkey       AS customer_nation_key,
    n.n_name            AS customer_nation_name,
    n.n_regionkey       AS customer_region_key,
    r.r_name            AS customer_region_name,
    c.c_phone           AS customer_phone,
    c.c_acctbal         AS customer_acctbal,
    c.c_mktsegment      AS customer_market_segment,
    c.c_comment         AS customer_comment
FROM {CATALOG}.{SILVER_SCHEMA}.customer c
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.nation n
    ON c.c_nationkey = n.n_nationkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.region r
    ON n.n_regionkey = r.r_regionkey
""")

_row_count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer").count()
print(f"dim_customer created: {_row_count} rows")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer").limit(10))

In [0]:
# ── dim_product: one row per product ──
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.dim_product AS
SELECT
    p.p_partkey          AS product_key,
    p.p_name             AS product_name,
    p.p_mfgr             AS product_manufacturer,
    p.p_brand            AS product_brand,
    p.p_type             AS product_type,
    p.p_size             AS product_size,
    p.p_container        AS product_container,
    p.p_retailprice      AS product_retailprice,
    p.p_comment          AS product_comment
FROM {CATALOG}.{SILVER_SCHEMA}.part p
""")

_row_count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_product").count()
print(f"dim_product created: {_row_count} rows")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_product").limit(10))

In [0]:
# ── dim_supplier: one row per supplier, with geography ──
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.dim_supplier AS
SELECT
    s.s_suppkey          AS supplier_key,
    s.s_name             AS supplier_name,
    s.s_address          AS supplier_address,
    s.s_nationkey        AS supplier_nation_key,
    n.n_name             AS supplier_nation_name,
    n.n_regionkey        AS supplier_region_key,
    r.r_name             AS supplier_region_name,
    s.s_phone            AS supplier_phone,
    s.s_acctbal          AS supplier_acctbal,
    s.s_comment          AS supplier_comment
FROM {CATALOG}.{SILVER_SCHEMA}.supplier s
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.nation n
    ON s.s_nationkey = n.n_nationkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.region r
    ON n.n_regionkey = r.r_regionkey
""")

_row_count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_supplier").count()
print(f"dim_supplier created: {_row_count} rows")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_supplier").limit(10))

In [0]:
# ── fact_sales: one row per order line ──
# Includes all keys, measures, and denormalized attributes for SQL, dashboard, and Genie tasks.
# Customer geography and supplier geography are explicitly distinguished.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.fact_sales AS
SELECT
    -- ── Primary & foreign keys ──
    l.l_orderkey                                 AS order_key,
    l.l_linenumber                               AS order_line_number,
    o.o_custkey                                  AS customer_key,
    l.l_partkey                                  AS product_key,
    l.l_suppkey                                  AS supplier_key,

    -- ── Customer geography (distinct from supplier geography) ──
    cust_n.n_nationkey                           AS customer_nation_key,
    cust_n.n_name                                AS customer_nation_name,
    cust_n.n_regionkey                           AS customer_region_key,
    cust_r.r_name                                AS customer_region_name,

    -- ── Supplier geography ──
    supp_n.n_nationkey                           AS supplier_nation_key,
    supp_n.n_name                                AS supplier_nation_name,
    supp_n.n_regionkey                           AS supplier_region_key,
    supp_r.r_name                                AS supplier_region_name,

    -- ── Measures (per metric definitions) ──
    l.l_quantity                                 AS quantity,
    l.l_extendedprice                            AS extended_price,
    l.l_discount                                 AS discount_rate,
    l.l_tax                                      AS tax_rate,
    l.l_extendedprice * (1 - l.l_discount)        AS net_revenue,
    l.l_extendedprice                            AS gross_revenue,
    l.l_extendedprice * (1 - l.l_discount) * l.l_tax  AS tax_amount,
    l.l_extendedprice * (1 - l.l_discount) * (1 + l.l_tax) AS total_amount,

    -- ── Order attributes (from orders) ──
    o.o_orderdate                                AS order_date,
    YEAR(o.o_orderdate)                          AS order_year,
    o.o_orderstatus                              AS order_status,
    o.o_orderpriority                            AS order_priority,
    o.o_totalprice                               AS order_total_price,
    o.o_shippriority                             AS order_ship_priority,
    o.o_clerk                                    AS order_clerk,

    -- ── Customer attributes (denormalized for Genie/dashboard) ──
    c.c_name                                     AS customer_name,
    c.c_mktsegment                               AS customer_market_segment,

    -- ── Product attributes (denormalized) ──
    p.p_name                                     AS product_name,
    p.p_brand                                    AS product_brand,
    p.p_type                                     AS product_type,

    -- ── Supplier attributes (denormalized) ──
    s.s_name                                     AS supplier_name,

    -- ── Shipping attributes ──
    l.l_shipdate                                 AS ship_date,
    l.l_commitdate                               AS commit_date,
    l.l_receiptdate                              AS receipt_date,
    l.l_shipmode                                 AS ship_mode,
    l.l_shipinstruct                             AS ship_instruct,
    l.l_returnflag                               AS return_flag,
    l.l_linestatus                               AS line_status,
    l.l_comment                                  AS line_comment
FROM {CATALOG}.{SILVER_SCHEMA}.lineitem l
JOIN {CATALOG}.{SILVER_SCHEMA}.orders o
    ON l.l_orderkey = o.o_orderkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.customer c
    ON o.o_custkey = c.c_custkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.part p
    ON l.l_partkey = p.p_partkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.supplier s
    ON l.l_suppkey = s.s_suppkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.nation cust_n
    ON c.c_nationkey = cust_n.n_nationkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.region cust_r
    ON cust_n.n_regionkey = cust_r.r_regionkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.nation supp_n
    ON s.s_nationkey = supp_n.n_nationkey
LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.region supp_r
    ON supp_n.n_regionkey = supp_r.r_regionkey
""")

_row_count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales").count()
print(f"fact_sales created: {_row_count:,} rows")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales").limit(10))

In [0]:
# ── vw_annual_sales_kpis: annual revenue, orders, active customers, avg order value ──
# Metric definitions:
#   Net revenue = SUM(l_extendedprice × (1 − l_discount))
#   Orders      = COUNT(DISTINCT o_orderkey)
#   Active customers = COUNT(DISTINCT customers with ≥1 order in the period)
#   Avg order value  = Net revenue ÷ distinct orders
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_annual_sales_kpis AS
SELECT
    order_year                                       AS sales_year,
    ROUND(SUM(net_revenue), 2)                                AS net_revenue,
    COUNT(DISTINCT order_key)                         AS total_orders,
    COUNT(DISTINCT customer_key)                      AS active_customers,
    ROUND(SUM(net_revenue) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM {CATALOG}.{GOLD_SCHEMA}.fact_sales
GROUP BY order_year ORDER BY sales_year
""")

print(f"vw_annual_sales_kpis created")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.vw_annual_sales_kpis"))

In [0]:
# ── Reconciliation: Gold vs Silver ──
# Independent calculation from Silver: SUM(l_extendedprice * (1 - l_discount)) and COUNT(*)
# Gold: SUM(net_revenue) and COUNT(*) from fact_sales

silver_metrics = spark.sql(f"""
    SELECT
        SUM(l_extendedprice * (1 - l_discount))   AS silver_net_revenue,
        COUNT(*)                                   AS silver_row_count
    FROM {CATALOG}.{SILVER_SCHEMA}.lineitem
""").collect()[0]

gold_metrics = spark.sql(f"""
    SELECT
        SUM(net_revenue)     AS gold_net_revenue,
        COUNT(*)             AS gold_row_count
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_sales
""").collect()[0]

silver_revenue = silver_metrics["silver_net_revenue"]
gold_revenue   = gold_metrics["gold_net_revenue"]
silver_rows    = silver_metrics["silver_row_count"]
gold_rows      = gold_metrics["gold_row_count"]

revenue_diff = float(gold_revenue or 0) - float(silver_revenue or 0)
row_diff     = (gold_rows or 0) - (silver_rows or 0)

revenue_pass = abs(revenue_diff) < 0.01
row_pass     = row_diff == 0

print("═" * 70)
print("  GOLD vs SILVER RECONCILIATION REPORT")
print("═" * 70)
print(f"  Net Revenue (Silver):  ${silver_revenue:>20,.2f}")
print(f"  Net Revenue (Gold):    ${gold_revenue:>20,.2f}")
print(f"  Revenue Difference:    ${revenue_diff:>20,.2f}  {'✅ PASS' if revenue_pass else '❌ FAIL'}")
print(f"  ──────────────────────────────────────────────")
print(f"  Row Count (Silver):    {silver_rows:>20,}")
print(f"  Row Count (Gold):      {gold_rows:>20,}")
print(f"  Row Difference:        {row_diff:>20,}  {'✅ PASS' if row_pass else '❌ FAIL'}")
print("═" * 70)
overall = "✅ PASS" if (revenue_pass and row_pass) else "❌ FAIL"
print(f"  Overall Status:        {overall}")
print("═" * 70)

In [0]:
# ── Dimension Key Uniqueness Validation ──
# Validate that the primary key is unique in each dimension table.

dim_checks = [
    ("dim_geography", "geography_key"),
    ("dim_customer",  "customer_key"),
    ("dim_product",   "product_key"),
    ("dim_supplier",  "supplier_key"),
]

print("═" * 70)
print("  DIMENSION KEY UNIQUENESS VALIDATION")
print("═" * 70)
sep20 = "─" * 20
sep12 = "─" * 12
sep14 = "─" * 14
sep10 = "─" * 10
print(f"  {'Table':<20} {'Key Column':<20} {'Total Rows':>12} {'Distinct Keys':>14} {'Status':>10}")
print(f"  {sep20} {sep20} {sep12} {sep14} {sep10}")

all_pass = True
for table_name, key_col in dim_checks:
    total_rows = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{table_name}").count()
    distinct_keys = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{table_name}").select(key_col).distinct().count()
    is_unique = total_rows == distinct_keys
    status = "✅ PASS" if is_unique else "❌ FAIL"
    if not is_unique:
        all_pass = False
    print(f"  {table_name:<20} {key_col:<20} {total_rows:>12,} {distinct_keys:>14,} {status:>10}")

print("═" * 70)
overall = "✅ ALL DIMENSIONS PASS" if all_pass else "❌ UNIQUENESS FAILURE DETECTED"
print(f"  Overall: {overall}")
print("═" * 70)